In [71]:
from torch.nn.functional import embedding
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [72]:
from youtube_rag.fetch_data import fetch_podcast_list_from_channel, VideoData
from dataclasses import asdict
import polars as pl

cc_details = pl.read_csv("../data/print/cut_the_clutter_podcast.csv").to_dicts()
cc_details = [VideoData(**c) for c in cc_details]

In [73]:
len(cc_details)

1974

In [74]:
# from youtube_rag.plot import plot_duration_as_scatter
# fig = plot_duration_as_scatter(cc_details)
# fig.update_yaxes(type="log")
# fig.update_xaxes(type="log")
# fig.show()

In [75]:
import numpy as np
#
# d = np.quantile(np.array([c.duration for c in cc_details if c.duration is not None]), 0.999)
# d
# [v for v in cc_details if v.duration is not None and v.duration >= d]

In [76]:
# cc_details[15].title
# cc_details[0].title, cc_details[0].video_id
cc_details[2].title

'India’s Rs 10,000 crore nudge to build container capacity & counter China'

In [77]:
from youtube_rag.fetch_data import remove_initial_ads_in_transcript

transcript_raw = f"../data/print/transcript/{cc_details[2].video_id}.json"

text = remove_initial_ads_in_transcript(transcript_raw)
# text = re.sub(r"\[music\]", "", text, flags=re.IGNORECASE)
# print(text)

In [78]:
from collections import Counter
import spacy
nlp = spacy.load("en_core_web_sm")

# 1. Tokenize and normalize each sentence using spaCy
doc = nlp(text)

sentences = [sent.text.strip() for sent in doc.sents]

cleaned_sentences = []

In [79]:
for sentence in sentences:
    sent_doc = nlp(sentence)

    tokens = [
        token.lemma_.lower()
        for token in sent_doc
        if token.is_alpha and not token.is_stop and not token.is_punct
    ]

    cleaned_sentences.append(" ".join(tokens))

tokens_per_sentence = [
    cleaned_sentences[0].strip().split(' ') for sentence in cleaned_sentences
]

# --------------------------------------------------
# 2. Word-frequency scoring
# --------------------------------------------------

word_counts = Counter(
    word
    for tokens in tokens_per_sentence
    for word in tokens
)

frequency_scores = []

for tokens in tokens_per_sentence:
    if tokens:
        score = sum(word_counts[word] for word in tokens) / len(tokens)
    else:
        score = 0.0

    frequency_scores.append(score)


In [80]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from youtube_rag.relevance import Relevance
from sklearn.feature_extraction.text import TfidfVectorizer

# tfidf_matrix
vectorizer = TfidfVectorizer(max_df=0.85)

tfidf_matrix = vectorizer.fit_transform(cleaned_sentences)


# Similarity between all pairs of sentences
similarity_matrix = cosine_similarity(tfidf_matrix)

relevance = Relevance.from_tfidf_matrix(tfidf_matrix)

relevance.shape, len(sentences)

((197,), 197)

In [81]:
# summary using tfidf relevance and similarity matrix
from youtube_rag.summarization import mmr_summary

summary = mmr_summary(
    sentences,
    relevance,
    similarity_matrix,
    top_k=3,
    lambda_param=0.7
)

for i, sentence in zip(summary[0], summary[1]):
    print(f"score {i:.4f}, {sentence}")


score 0.9985, Now, we are talking about this one because Government of India just announced cabinet just this Wednesday took a decision to offer a 10,000 crore rupees support divided in various in in divided under various heads for for container building in India going up to 2039-40.
score 0.9414, To understand this, let me let me try and simplify this because the whole idea of cut the clutter is to simplify and it's because you are so forgiving with me that I pick up sometimes really what sound like really boring subjects to talk about because these are also interesting, these are important, these are very vital.
score 1.0000, In fact, some of the money the government has allocated out of this 10,000 crores is going towards quality control and testing facilities because these boxes have to survive years and years of exposure, heavy loads, rough handling.


In [82]:
from youtube_rag.summarization import network_summary

net_score, net_sentences = network_summary(sentences, similarity_matrix, top_k=3)

for s, sen in zip(net_score, net_sentences):
    print(round(s, 3), sen)

0.014 After all, what's a container?
0.015 So, India needs a lot of containers.
0.014 That is unless you have more containers.


In [83]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# Encode the sentences
embeddings = model.encode(sentences, normalize_embeddings=True)

passage_embedding = model.encode([" ".join(sentences)], normalize_embeddings=True)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3186.86it/s]


In [84]:
from youtube_rag.summarization import semantic_summary
# Calculate similarity between sentences
# similarity_matrix = cosine_similarity(embeddings)
sem_similarity_matrix = cosine_similarity(embeddings)

# sentences and passage similarity
sem_relevance = Relevance.from_embeddings(embeddings, passage_embedding)

summary = semantic_summary(sentences, embeddings)

for score, sen in zip(summary[0], summary[1]):
    print(score, sen)


0.722 And the idea is to increase India's container building capacity 10 from the current 70,000 or there abouts to more than 7 and a half lakh per year.
0.763 So, India needs a lot of containers.
0.73 India right now imports about 2 million containers per year.


In [94]:
summary = mmr_summary(sentences, sem_relevance, sem_similarity_matrix, top_k=5, lambda_param=0.7)

for i, sentence in zip(summary[0], summary[1]):
    print(f"score {i:.4f}, {sentence}")

score 0.6355, In the middle of all of this, why are we focusing on a humble piece of steel, humble box of steel, humble box of steel called a container, a shipping container, a trucking container, a railway container?
score 0.7043, Now, we are talking about this one because Government of India just announced cabinet just this Wednesday took a decision to offer a 10,000 crore rupees support divided in various in in divided under various heads for for container building in India going up to 2039-40.
score 0.7211, And the idea is to increase India's container building capacity 10 from the current 70,000 or there abouts to more than 7 and a half lakh per year.
score 0.6414, If you look at India's trade, for example, 95% of India's trade by volume is carried out by ships, almost all of it by containers.
score 0.6026, India also has now proudly double-decker container railways, long really long double-decker container railways.


In [86]:
# relevance with centrality based relevance
from youtube_rag.summarization import mmr_summary

centrality = Relevance.from_similarity_matrix_for_centrality(sem_similarity_matrix)

summary = mmr_summary(
    sentences,
    centrality,
    sem_similarity_matrix,
    top_k=3,
    lambda_param=0.7
)

for i, sentence in zip(summary[0], summary[1]):
    print(f"score {i:.4f}, {sentence}")

score 0.3151, So, India needs a lot of containers.
score 0.1562, It's a very basic box of steel, specialized steel, but a basic box of steel.
score 0.2252, If they add 25,000, then they get the benefits of a of a brownfield TU manufacturing company or container manufacturing company.


In [92]:
from youtube_rag.summarization import summary_facility_location


# summary = summary_facility_location(sentences, sem_similarity_matrix, centrality, top_k=5)
summary = summary_facility_location(sentences, sem_similarity_matrix, sem_similarity_matrix, top_k=5)

# summary = mmr_summary_facility_location(sentences, similarity_matrix, top_k=5)

for i, sentence in enumerate(summary):
    print(i, sentence)

0 Usually, it's a 20-ft long box of horizontal box of steel or what is called as a TEU.
1 A container, if at all, costs just about $2,000.
2 So India needs to send out more manufactured goods.
3 So, India needs a lot of containers.
4 India has a large import bill.
5 Global boom in demand, suddenly containers were not available.
6 So, generally for now, the standard 20-ft TEU will get 49,500 rupees.
7 Those containers then have to be ferried by ships to the other side.
8 Also, sea breeze is highly corrosive.
9 If they add 25,000, then they get the benefits of a of a brownfield TU manufacturing company or container manufacturing company.


In [ ]:
# semantic search with sentence transformers
corpus_embeddings = model.encode_document(sentences, convert_to_tensor=True)

# Query sentences:
queries = [
    # "Any published memoir or book?",
    "Is there any book used as reference?"
]


In [36]:
# for s in sentences:
#     if 'book' in s.lower() or 'memoir' in s.lower() or 'novel' in s.lower():
#         print(s)

In [97]:
from sentence_transformers.cross_encoder import CrossEncoder

# Pre-trained cross encoder
cross_encoder = CrossEncoder("cross-encoder/stsb-distilroberta-base")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 11567.61it/s]


In [98]:
# 1. We rank all sentences in the corpus for the query
ranks = cross_encoder.rank(queries[0], sentences)

# Print the scores
print("Query:", queries[0])
for rank in ranks[:3]:
    print(f"{rank['score']:.2f}\t{sentences[rank['corpus_id']]}")

Query: Is there any book used as reference?
0.40	A container, because the container is locked, it's not as if theft does not take place from a container or pilferage does not take place from a container, but with a container, it still takes some doing to break it open or to unlock it because containers can be locked.
0.38	Their corners have to be fit They their corners have to fit one on top of the other because if they don't and they if they put one on top of the other even a loose one loose one can can damage the ship or in fact in fact threaten the ship.
0.37	However, it needs some specialized things which I will tell you about, but before that, let me explain to you what is called container container Let me explain to you what is called container turnaround or repositioning that is integral to shipping logistics, railway logistics because now you see containers on your rails also.


In [95]:
# pip install bm25s
import bm25s

# Create the BM25 model and index the corpus
retriever = bm25s.BM25(corpus=sentences)
retriever.index(bm25s.tokenize(sentences))

# Query the corpus and get top-k results
query = ["Any published memoir or book?"]
results, scores = retriever.retrieve(bm25s.tokenize(query), k=1)

# Let's see what we got!
doc, score = results[0, 0], scores[0, 0]
print(f"Rank {1} (score: {score:.2f}): {doc}")


Rank 1 (score: 2.84): Then, if you don't have any of that, without any of that, a horse is just another head of cattle that doesn't give you any milk, right?


In [29]:
# Find the closest 5 sentences of the corpus for each query sentence based on cosine similarity
# import torch
#
# top_k = min(3, len(sentences))
# for query in queries:
#     query_embedding = model.encode_query(query, convert_to_tensor=True)
#
#     # We use cosine-similarity and torch.topk to find the highest 5 scores
#     similarity_scores = model.similarity(query_embedding, corpus_embeddings)[0]
#     scores, indices = torch.topk(similarity_scores, k=top_k)
#
#     print("\nQuery:", query)
#     print("Top 5 most similar sentences in corpus:")
#
#     for score, idx in zip(scores, indices):
#         print(f"(Score: {score:.4f})", sentences[idx])